# __Demo: Prompt Patterns for Engineering Tasks with Hugging Face__

## __Steps to Perform:__
Step 1: Set up the Environment and Initialize `Qwen/Qwen2.5-0.5B-Instruct`

Step 2: Define a Function to Get Completion

Step 3: Pattern 1 -- Code Generation

Step 4: Pattern 2 -- Test Design

Step 5: Pattern 3 -- PR Review

Step 6: Pattern 4 -- Documentation

Step 7: Pattern 5 -- Deployment Scripts

### __Step 1: Set up the Environment and Initialize `Qwen/Qwen2.5-0.5B-Instruct`__
- Installs required libraries (`transformers`, `torch`, `accelerate`).
- Loads `Qwen/Qwen2.5-0.5B-Instruct` onto the GPU in 16-bit half precision (`torch.float16`).
- No API key, token, or billing configuration is required.

In [1]:
!pip install -q transformers torch accelerate

import os
import torch
from transformers import pipeline

# Initialize text-generation pipeline on GPU with float16
generator = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-0.5B-Instruct",
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)
print("✅ Qwen/Qwen2.5-0.5B-Instruct loaded successfully on GPU.")

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

✅ Qwen/Qwen2.5-0.5B-Instruct loaded successfully on GPU.


### __Step 2: Define a Function to Get Completion__
- Sends the prompt to the Hugging Face pipeline formatted as an instruction chat message.
- Sets `do_sample=False` for deterministic (greedy) software engineering completions.

In [2]:
def get_completion(prompt, max_tokens=250):
    messages = [{"role": "user", "content": prompt}]
    output = generator(messages, max_new_tokens=max_tokens, do_sample=False)
    return output[0]["generated_text"][-1]["content"].strip()

print("✅ Completion helper ready.")

✅ Completion helper ready.


### __Step 3: Pattern 1 -- Code Generation__
Describe the function's signature, constraints, and one example input/output before asking for an implementation -- this is the reusable pattern for every code-generation prompt in this library.

In [3]:
code_gen_prompt = """
Write a Python function with this signature: def parse_severity(log_line: str) -> str.
Constraints: it must return one of "CRITICAL", "ERROR", "WARN", or "INFO" based on the first matching keyword found in the log line (case-insensitive); return "UNKNOWN" if none match.
Example: parse_severity("2026-09-10 ERROR failed to connect") should return "ERROR".
Return only the function code.
"""
print("=== Pattern 1: Code Generation ===")
print(get_completion(code_gen_prompt))


=== Pattern 1: Code Generation ===
```python
import re

def parse_severity(log_line: str) -> str:
    """
    Parses a log line and returns the severity level as a string.

    Args:
        log_line (str): The log line to be parsed.

    Returns:
        str: The severity level ('CRITICAL', 'ERROR', 'WARN', 'INFO'), or 'UNKNOWN' if no match is found.
    """
    # Regular expression pattern for matching severity levels
    severity_pattern = r'\b(CRITICAL|ERROR|WARN|INFO)\b'
    
    # Search for the first occurrence of the regex pattern in the log line
    match = re.search(severity_pattern, log_line)
    
    if match:
        return match.group(0).lower()
    else:
        return "UNKNOWN"

# Check function to verify the correctness of the generated function
def check():
    test_cases = [
        ("2026-09-10 ERROR failed to connect", "ERROR"),
        ("2026-09-10 CRITICAL system failure", "CRITICAL"),
        ("2026-09-10 WARN critical error", "WARN"),
        ("2026


### __Step 4: Pattern 2 -- Test Design__
Provide the function and ask for unit tests, edge cases, and the specific assertions expected for each.

In [4]:
test_design_prompt = """
Write pytest unit tests for this function, covering the normal case, the case where no keyword matches, and a case-insensitivity check. State the specific assertion expected for each test.

def parse_severity(log_line: str) -> str:
    for level in ["CRITICAL", "ERROR", "WARN", "INFO"]:
        if level in log_line.upper():
            return level
    return "UNKNOWN"
"""
print("=== Pattern 2: Test Design ===")
print(get_completion(test_design_prompt))


=== Pattern 2: Test Design ===
To write comprehensive unit tests for the `parse_severity` function, we need to cover both the normal case scenario (where all keywords match), as well as cases where there is no matching keyword, and also ensure that the function behaves correctly when the input string is case-insensitive. Here's how you can structure your tests:

```python
import pytest

# Define the function to be tested
def parse_severity(log_line: str) -> str:
    # Dictionary mapping keywords to their corresponding severity levels
    severity_levels = {
        "CRITICAL": "CRITICAL",
        "ERROR": "ERROR",
        "WARN": "WARN",
        "INFO": "INFO",
        "UNKNOWN": "UNKNOWN"
    }
    
    # Check for the normal case scenario
    if log_line.lower() in [level.lower() for level in severity_levels]:
        return severity_levels[log_line.lower()]
    
    # Case-insensitive check
    elif log_line.isupper():
        return severity_levels.get(log_line.upper(), "UNKNOWN")


### __Step 5: Pattern 3 -- PR Review__
Supply the diff and ask the model to flag correctness, security, and style issues separately, citing line numbers.

In [5]:
pr_review_prompt = """
Review this diff. Report correctness issues, security issues, and style issues as three separate lists, citing the line number for each.

1  def parse_severity(log_line):
2      for level in ["CRITICAL", "ERROR", "WARN", "INFO"]:
3          if level in log_line:
4              return level
5      return "UNKNOWN"
"""
print("=== Pattern 3: PR Review ===")
print(get_completion(pr_review_prompt))


=== Pattern 3: PR Review ===
### Correctness Issues
1. **Line 2**: The function `parse_severity` is defined to take a single argument `log_line`, but it does not use any of its arguments directly within the function body.
2. **Line 6**: The condition inside the loop should be checked after the list comprehension has been executed, rather than before. This would ensure that all elements in the list are processed correctly.
3. **Line 7**: The indentation in the conditional statement should be consistent with the structure of the code. It should start with an empty line or a comment indicating the purpose of the conditional block.

### Security Issues
1. **Line 8**: The function name `parse_severity` is too short and could potentially lead to unexpected behavior when used in larger codebases. A more descriptive and meaningful function name might improve readability.
2. **Line 9**: The variable `level` is declared without initialization, which can lead to potential runtime errors if not pr

### __Step 6: Pattern 4 -- Documentation__
Give the function body and ask for a docstring covering arguments, return value, and exceptions -- nothing invented.

In [6]:
documentation_prompt = """
Write a Google-style docstring for this function, covering Args, Returns, and any exceptions it raises. Do not invent behavior that is not in the code.

def parse_severity(log_line: str) -> str:
    for level in ["CRITICAL", "ERROR", "WARN", "INFO"]:
        if level in log_line.upper():
            return level
        return "UNKNOWN"
"""
print("=== Pattern 4: Documentation ===")
print(get_completion(documentation_prompt))


=== Pattern 4: Documentation ===
```python
import re

def parse_severity(log_line: str) -> str:
    """
    Parses a log line to determine its severity level based on predefined levels.
    
    Parameters:
    - log_line (str): A string representing a log message with potential severity levels.
    
    Returns:
    - str: The determined severity level or 'UNKNOWN' if no match is found.
    
    Raises:
    - ValueError: If the input log_line does not contain a valid severity level.
    """
    # Regular expression pattern to match severity levels
    severity_pattern = r"(\w+)\s+\d+"

    # Search for the first occurrence of a severity level in the log line
    match = re.search(severity_pattern, log_line)
    if match:
        severity_level = match.group(1).upper()
        return severity_level
    else:
        raise ValueError("Invalid severity level in the log line.")
```

This docstring provides a clear description of what the function does, including the parameters it accepts,

### __Step 7: Pattern 5 -- Deployment Scripts__
State the target environment and constraints explicitly -- a generated deploy script run against the wrong environment is a production incident, so ambiguity here is not acceptable.

In [7]:
deployment_prompt = """
Write a bash script that deploys a Docker container to a staging environment (not production). Constraints: it must pull the image tagged "staging-latest", stop and remove any existing container named "alerts-api-staging" first, and print a confirmation message naming the environment it deployed to.
"""
print("=== Pattern 5: Deployment Scripts ===")
print(get_completion(deployment_prompt))


=== Pattern 5: Deployment Scripts ===
```bash
#!/bin/bash

# Pulling the latest staging image
docker pull staging-latest

# Stopping any existing container named alerts-api-staging
docker stop alerts-api-staging

# Removing the existing container
docker rm alerts-api-staging

# Creating a new container with the latest staging image
docker run -d --name alerts-api-staging \
  --restart=always \
  --env-file .env.staging \
  --volume /var/run/docker.sock:/var/run/docker.sock \
  --publish 8080:8080 \
  staging-latest

echo "Deployment successful! The container has been successfully deployed to the staging environment."
```

To use this script:
1. Save it as `deploy_staging.sh`.
2. Make the script executable by running `chmod +x deploy_staging.sh`.
3. Run the script using `./deploy_staging.sh`.

This script will ensure that you have a new container running in your staging environment, which is not part of the production setup. It will also clean up any old containers if they exist.


## __Conclusion__
All five patterns share the same underlying structure -- instruction, context, input data, output indicator -- adapted to what each stage of the engineering workflow actually needs to see. These five patterns are the ones carried into the next demo, where they are saved as a reusable personal prompt library.